In [ ]:
import os, io, random, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU non actif -> Settings > Accelerator > GPU"
print("device:", device, torch.cuda.get_device_name(0))


In [ ]:
BASE = Path("/kaggle/input")
comp = next(p.parent for p in BASE.rglob("sample_submission.csv"))

BACKBONE = "tf_efficientnetv2_s"
IMG = 448
BATCH = 24
EPOCHS = 6
LR = 3e-4
PRETRAINED = True
NFOLD = 5

def load_gt(path):
    df = pd.read_csv(path)
    return df.rename(columns={df.columns[1]: "classe"})

train_df = load_gt(comp / "train" / "ground_truth.csv")
val_df = load_gt(comp / "val" / "ground_truth.csv")
sub = pd.read_csv(comp / "sample_submission.csv")
LABEL_COL = [c for c in sub.columns if c != "image_id"][0]

test_imgs = comp / "test" / "images"


all_df = pd.concat([train_df.assign(src="train"), val_df.assign(src="val")], ignore_index=True)
path_of = {}
for _, r in all_df.iterrows():
    path_of[r.image_id] = comp / r.src / "images" / r.image_id
lbl = dict(zip(all_df.image_id, all_df.classe))
y_all = all_df.classe.values

folds = list(StratifiedKFold(NFOLD, shuffle=True, random_state=SEED).split(all_df, y_all))
print("total:", len(all_df), "| test:", len(sub), "| folds:", NFOLD)


In [ ]:
def ela(pil, q=90, scale=12):
    buf = io.BytesIO()
    pil.save(buf, "JPEG", quality=q); buf.seek(0)
    re = Image.open(buf).convert("RGB")
    d = np.abs(np.asarray(pil, np.int16) - np.asarray(re, np.int16))
    return np.clip(d * scale, 0, 255).astype(np.uint8)

def noise(pil, scale=4):
    a = np.asarray(pil, np.float32)
    res = a - cv2.GaussianBlur(a, (0, 0), 1.0)
    return np.clip(res * scale + 128, 0, 255).astype(np.uint8)

def forensic(pil):
    return np.concatenate([ela(pil), noise(pil)], axis=2)

rgb_fn = lambda im: np.asarray(im)


def build_cache(id_to_path, ids, fn):
    def make(name):
        with Image.open(id_to_path[name]) as im:
            im = im.convert("RGB"); im.thumbnail((IMG, IMG))
            return name, fn(im)
    with ThreadPoolExecutor(max_workers=8) as ex:
        return dict(ex.map(make, list(ids)))


class Imgs(Dataset):
    def __init__(self, ids, cache, tf, labels=None):
        self.ids = list(ids); self.cache = cache; self.tf = tf; self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        name = self.ids[i]
        x = self.tf(image=self.cache[name])["image"]
        if self.labels is None:
            return x, name
        return x, torch.tensor(self.labels[name], dtype=torch.float32)


def transforms(nch):
    mean, std = [0.5] * nch, [0.5] * nch
    aug = A.Compose([
        A.PadIfNeeded(IMG, IMG, border_mode=0),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
        A.Normalize(mean, std, max_pixel_value=255.0), ToTensorV2(),
    ])
    ev = A.Compose([
        A.PadIfNeeded(IMG, IMG, border_mode=0),
        A.Normalize(mean, std, max_pixel_value=255.0), ToTensorV2(),
    ])
    return aug, ev

dl_args = dict(num_workers=4, pin_memory=True, persistent_workers=True)


In [ ]:
import copy

@torch.no_grad()
def predict(model, dl, tta=False):
    model.eval()
    out = []
    for x, _ in dl:
        x = x.to(device)
        views = [x, torch.flip(x, [3]), torch.flip(x, [2])] if tta else [x]
        with torch.amp.autocast("cuda"):
            p = sum(torch.sigmoid(model(v).squeeze(1)) for v in views) / len(views)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)


def train_fold(tr_ids, va_ids, cache, tf_tr, tf_ev, in_chans, tag):
    tr_dl = DataLoader(Imgs(tr_ids, cache, tf_tr, lbl), BATCH, shuffle=True, drop_last=True, **dl_args)
    va_dl = DataLoader(Imgs(va_ids, cache, tf_ev), BATCH, shuffle=False, **dl_args)
    va_y = np.array([lbl[n] for n in va_ids])

    model = timm.create_model(BACKBONE, pretrained=PRETRAINED, num_classes=1, in_chans=in_chans).to(device)
    pw = (y_all == 0).sum() / max((y_all == 1).sum(), 1)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=EPOCHS, steps_per_epoch=len(tr_dl))
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_state = 0.0, None
    for epoch in range(EPOCHS):
        model.train()
        for x, yy in tr_dl:
            x, yy = x.to(device), yy.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(x).squeeze(1), yy)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
        f1 = f1_score(va_y, (predict(model, va_dl) > 0.5).astype(int))
        if f1 > best_f1:
            best_f1, best_state = f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f"  [{tag}] best fold f1={best_f1:.4f}")
    return model


In [ ]:
def run_representation(fn, in_chans, tag):
    cache = build_cache(path_of, all_df.image_id, fn)
    test_cache = build_cache({n: test_imgs / n for n in sub.image_id}, sub.image_id, fn)
    tf_tr, tf_ev = transforms(in_chans)

    oof = np.zeros(len(all_df))
    test_pred = np.zeros(len(sub))
    ids = all_df.image_id.values
    for k, (tr_idx, va_idx) in enumerate(folds):
        print(f"[{tag}] fold {k}")
        model = train_fold(ids[tr_idx], ids[va_idx], cache, tf_tr, tf_ev, in_chans, tag)
        oof[va_idx] = predict(model, DataLoader(Imgs(ids[va_idx], cache, tf_ev), BATCH, **dl_args), tta=True)
        test_pred += predict(model, DataLoader(Imgs(sub.image_id, test_cache, tf_ev), BATCH, **dl_args), tta=True) / NFOLD
        del model; gc.collect(); torch.cuda.empty_cache()

    f1 = f1_score(y_all, (oof > 0.5).astype(int))
    print(f"[{tag}] OOF f1@0.5={f1:.4f}")
    del cache, test_cache; gc.collect(); torch.cuda.empty_cache()
    return oof, test_pred


In [ ]:
oof_rgb, test_rgb = run_representation(rgb_fn, 3, "rgb")
oof_for, test_for = run_representation(forensic, 6, "forensic")


In [ ]:
grid = np.linspace(0.05, 0.95, 181)

for name, o in [("rgb", oof_rgb), ("forensic", oof_for)]:
    print(f"{name:9} OOF f1@0.5={f1_score(y_all, (o > 0.5).astype(int)):.4f}")


best = (0, 0.5, 0.5)
for w in np.linspace(0.3, 0.7, 9):
    ens = w * oof_for + (1 - w) * oof_rgb
    sc = [f1_score(y_all, (ens > t).astype(int)) for t in grid]
    if max(sc) > best[0]:
        best = (max(sc), w, grid[int(np.argmax(sc))])
BEST_F1, W, T = best
print(f"\nensemble OOF f1={BEST_F1:.4f}  poids_forensique={W:.2f}  seuil={T:.3f}")


In [ ]:
test_ens = W * test_for + (1 - W) * test_rgb
out = sub[["image_id"]].copy()
out[LABEL_COL] = (test_ens > T).astype(int)
assert out[LABEL_COL].isin([0, 1]).all() and out.image_id.is_unique
out.to_csv("submission.csv", index=False)
print(out[LABEL_COL].value_counts())
